# Yatharth Music AI — Free GPU + temporary public link

This notebook starts ACE-Step 1.5, connects the Yatharth Music AI FastAPI app, and creates a temporary public link for testing.

**Important:** free Colab GPU availability and runtime duration are not guaranteed. The public link disappears when the runtime stops. Do not use this as permanent hosting.

In [ ]:
!nvidia-smi || true
!rm -rf /content/ACE-Step-1.5 /content/yatharth-music-ai
!git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git /content/ACE-Step-1.5
!git clone --depth 1 https://github.com/rampaulsaini/yatharth-music-ai.git /content/yatharth-music-ai
%cd /content/ACE-Step-1.5
!pip -q install uv
!uv sync --frozen
%cd /content/yatharth-music-ai
!pip -q install -r requirements.txt

In [ ]:
# Start ACE-Step REST API. ACE-Step automatically adapts LLM usage to available GPU VRAM.
import subprocess, time, os, requests
log = open('/content/acestep.log', 'w')
proc = subprocess.Popen(['uv','run','python','-m','acestep.api_server','--host','127.0.0.1','--port','8001'], stdout=log, stderr=subprocess.STDOUT, cwd='/content/ACE-Step-1.5')
time.sleep(20)
print('ACE-Step process:', proc.poll(), 'PID:', proc.pid)
try:
    r = requests.get('http://127.0.0.1:8001/health', timeout=10)
    print('ACE-Step health:', r.status_code, r.text[:1000])
except Exception as e:
    print('ACE-Step is still starting. Check /content/acestep.log if needed:', e)

In [ ]:
# Start Yatharth Music AI in real-AI mode on port 8000.
# Cloudflare tunnel connects locally, so Uvicorn is explicitly told to trust
# forwarded headers only from the local tunnel process. This lets FastAPI
# preserve each visitor's client IP for task ownership/rate limiting without
# enabling TRUST_PROXY inside the application to blindly trust user headers.
import subprocess, time, requests, os
ylog = open('/content/yatharth.log', 'w')
env = os.environ.copy()
env['DEMO_MODE'] = 'false'
env['MUSIC_ENGINE_URL'] = 'http://127.0.0.1:8001'
env['TRUST_PROXY'] = 'false'
backend = subprocess.Popen(['python','-m','uvicorn','main:app','--host','0.0.0.0','--port','8000','--proxy-headers','--forwarded-allow-ips','127.0.0.1'], stdout=ylog, stderr=subprocess.STDOUT, cwd='/content/yatharth-music-ai', env=env)
time.sleep(5)
r = requests.get('http://127.0.0.1:8000/api/health', timeout=10)
print('Yatharth health:', r.status_code, r.json())
print('Backend PID:', backend.pid)

In [ ]:
# Create a temporary public HTTPS link without requiring a Hugging Face account.
import subprocess, re, time
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
tunnel_log = open('/content/cloudflared.log', 'w')
tunnel = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8000'], stdout=tunnel_log, stderr=subprocess.STDOUT)
public_url = None
for _ in range(30):
    time.sleep(2)
    try:
        text = open('/content/cloudflared.log', errors='ignore').read()
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
        if match:
            public_url = match.group(0)
            break
    except FileNotFoundError:
        pass
print('YATHARTH PUBLIC LINK:', public_url or 'Not ready yet; inspect /content/cloudflared.log')
print('Keep this Colab runtime running while using the link.')

## Test

Open the **YATHARTH PUBLIC LINK** printed above on your phone or computer. Generate a short song first.

The backend now uses Uvicorn's trusted-proxy handling for the local Cloudflare tunnel. This keeps task ownership and IP-based rate limiting tied to the visitor IP instead of the shared local proxy address.

If ACE-Step is still loading, wait a little and refresh. If there is an error, run the next diagnostic cell.

In [ ]:
print('--- ACE-Step log ---')
print(open('/content/acestep.log', errors='ignore').read()[-6000:])
print('--- Yatharth log ---')
print(open('/content/yatharth.log', errors='ignore').read()[-6000:])
print('--- Cloudflare log ---')
print(open('/content/cloudflared.log', errors='ignore').read()[-3000:])